# 03 · Gerçek NAS: one-shot süper ağ eğitimi (`nas_super1`)

**Karar 6 (Furkan, 2026-10-16), dondurulmuş tasarım.** Danışman görüşmesinde ayrık ızgara araması (nas_r1b, nas_r2) yetersiz bulundu → ağırlık paylaşımlı süper ağ NAS. Bu notebook yalnız süper ağı eğitir; arama 03b, 10 adaylık sıfırdan 60 epoch sıralama referansı 03c, final + Nano 300 epoch × 3 tohum 03d.

**Arama uzayı** (`exps/nas/supernet.py`; genişlik yok): her depthwise katmanda kernel {3, 5, 7} (30 katman) · dark3/dark4 CSP blok sayısı {1, 2, 3} · çözünürlük 320–512 (adım 32) → ≈ 1,3 × 10¹⁶ alt ağ. Tek 7×7 ağırlık, alt kernel merkez kesit; başlangıçta her alt ağ önceden eğitilmiş Nano ile aynı çıktıyı verir.

**Eğitim:** tam train2017 (önceden eğitilmiş Nano ile aynı veri), batch 64, fp16, çok ölçekli, lr 0,001/64, ısınma 1 epoch, son 2 epoch mosaic'siz, EMA. **Sandwich (BigNAS):** her iterasyonda en büyük + en küçük + 2 rastgele alt ağ, kayıplar ortalanır, tek adım. **BigNAS'tan sapma:** inplace distillation yok (açık uyarlama). **60 epoch ilk aşama**; uzatma en fazla 1 kez, 30 epoch, `uzatma_tepe_lr = 0,5 × ilk_aşama_tepe_lr`, yalnız manifestte önceden yazılı ölçütle ve Furkan onayıyla.

**Veri ayrımı (sızıntı önlemi):** önceden eğitilmiş Nano train2017'yi gördüğü için arama val2017-A'da (2500; uygunluk A_fit1k), BN kalibrasyonu train2017 bn2k'da. **val2017-B (2500) mühürlü:** NAS + budama + nicemleme yöntem kararları bitene kadar açılmaz. Not: B önçalışmada (nas_r1b/nas_r2, tam val2017) dolaylı kullanıldı. Nihai değerlendirme: gönderim kabulü doğrulanırsa COCO test-dev, yoksa B.

**İzleme:** 50 alt ağlık sabit panel (gecikme vekiline göre 5 dilim × 10) + 4 referans alt ağ, epoch 0/20/30/40/50/60'ta BN kalibreli fit1k mAP (0–100). Erken kontrol: epoch 4'te `nano_416` düşüşü ≤ 3,0 puan.

**Kesintiye dayanıklılık:** tam durum (ham + EMA, optimizer, AMP ölçekleyici, RNG, epoch/iter) her 300 iterasyonda ve epoch sonunda **doğrulanmış depoya** (benzersiz dosya → boyut + sha256 + yükleme denetimi → `.ok.json`; önceki sağlam kayıt silinmez). Veri sırası, artırmalar (örnek başına: koşu tohumu + epoch + mantıksal konum) ve çok ölçek tohumlu → devam birebir aynı akış. Hücre 7 bunu GPU'da kanıtlar: kesintisiz koşu ile (a) kayıt yazılırken öldürülen, (b) eğitim ortasında `kill -9` edilen koşunun iterasyon izleri karşılaştırılır.

**Uzun koşu kuralı:** hücre 1–8 doğrulama/pilot/manifest; **hücre 9 Furkan'ın başlatma onayıyla.** Uzun koşudan önce gereken kanıtlar: dondurulmuş manifest, başarılı kesinti/kurtarma testi, pilot maliyet ölçümü, test-dev erişim durumu (manifestte kayıtlı).

**Kesintide:** runtime'ı yeniden bağla → hücre 1–8 (hücre 4'te Drive elle; 6–8 kayıtlıysa hızla geçer) → 9 (en yeni doğrulanmış kayıttan kendiliğinden devam) / 10.

**Kayıtlar** (`Drive edge_ai/results/nas_super1/`; depoda `.pth` hariç): `manifest.json` · `splits_manifest.json` · `pilot.json` · `resume_test.json` · `panel.json` · `runs/supernet/{status.json, train_log.txt, ckpt/}` · `snapshots.csv` · `panel_stats.json` · `nas_super1_summary.json`.

Kod: `exps/nas/supernet.py`, `scripts/{nas_data, ckpt_store, supernet_search, make_splits, check_supernet, check_resume}.py`.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
NAS_EXPS = f"{WS}/exps/nas"
if NAS_EXPS not in sys.path:
    sys.path.insert(0, NAS_EXPS)
import importlib, nas_search as ns, nas_data as nd, supernet as sn, supernet_search as ss
for _m in (ns, nd, sn, ss):
    importlib.reload(_m)
EXP_FILE = f"{WS}/exps/nas/supernet.py"

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_super1"   # gerçek NAS, süper ağ (Karar 6, Furkan 2026-10-16)
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: tam COCO train2017 + val2017 (Colab yereline)
# train2017 görüntüleri Drive'da yok: Drive'da `data/train2017.zip` varsa oradan, yoksa resmi COCO sunucusundan
# Colab yereline indirilir (≈18 GB). Drive kotası için zip varsayılan olarak Drive'a KAYDEDİLMEZ; her yeni
# runtime'da yeniden indirilir. Anotasyonlar ve val2017 P0'dan beri Drive'da.
# Pilot ve kesinti testi (hücre 7) için 10k alt kümenin yalnız anotasyon json'u açılır (görselleri train2017'de zaten var).
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
TRAIN_ZIP_DRIVE = f"{DRIVE_DATA}/train2017.zip"
TRAIN_ZIP_URL = "http://images.cocodataset.org/zips/train2017.zip"
SAVE_TRAIN_ZIP_TO_DRIVE = False   # True: ilk indirmede Drive'a da kopyalar (≈18 GB kota)
N_TRAIN = 118_287
for z in (ANN_ZIP, VAL_ZIP, SUBSET_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
with zipfile.ZipFile(SUBSET_ZIP) as z:
    if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"):
        z.extract("annotations/instances_train2017_10k.json", COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)

n_local = len(os.listdir(f"{COCO_LOCAL}/train2017")) if os.path.isdir(f"{COCO_LOCAL}/train2017") else 0
if n_local != N_TRAIN:
    local_zip = "/content/train2017.zip"
    if os.path.exists(TRAIN_ZIP_DRIVE):
        print("train2017.zip Drive'dan kopyalanıyor…", flush=True)
        shutil.copy(TRAIN_ZIP_DRIVE, local_zip)
        env["train_zip_source"] = "drive"
    else:
        print("train2017.zip resmi COCO sunucusundan indiriliyor (≈18 GB)…", flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "--progress=dot:giga", "-O", local_zip + ".part",
                        TRAIN_ZIP_URL], check=True)
        os.replace(local_zip + ".part", local_zip)
        env["train_zip_source"] = TRAIN_ZIP_URL
    env["train_zip_bytes"] = os.path.getsize(local_zip)
    env["train_zip_sha256"] = ns.sha256(local_zip)
    if SAVE_TRAIN_ZIP_TO_DRIVE and not os.path.exists(TRAIN_ZIP_DRIVE):
        shutil.copy(local_zip, TRAIN_ZIP_DRIVE + ".tmp")
        os.replace(TRAIN_ZIP_DRIVE + ".tmp", TRAIN_ZIP_DRIVE)
    print("açılıyor…", flush=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", COCO_LOCAL], check=True)
    os.remove(local_zip)
env["train_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["data_prep_s"] = round(time.time() - t0, 1)
env["disk_free_after_data"] = sh("df -h /content | tail -1")
print({k: env.get(k) for k in ("train_images", "val_images", "train_zip_source", "train_zip_sha256",
                               "train_ann_sha256", "data_prep_s", "disk_free_after_data")})
assert env["train_images"] == N_TRAIN and env["val_images"] == 5000

In [ ]:
# 6 · Veri ayrımları: val2017 A/B (A arama/geliştirme, B mühürlü), A_fit1k, train2017 bn2k
# Tohum sabit (2026) → her runtime'da aynı ayrım; kimlik listesi ve dosya sha256'ları Drive'daki ilk kayıtla eşleşmeli.
SPLIT_SEED = 2026
sm_local = f"{COCO_LOCAL}/annotations/nas_splits_manifest.json"
if not os.path.exists(sm_local):
    r = subprocess.run(["python", f"{WS}/scripts/make_splits.py", "--coco", COCO_LOCAL, "--seed", str(SPLIT_SEED)],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1500:]); assert r.returncode == 0
sm = json.load(open(sm_local))
sm_drive = ns.read_json(f"{DRIVE_RESULTS}/splits_manifest.json")
if sm_drive is None:
    ns.write_json(f"{DRIVE_RESULTS}/splits_manifest.json", sm)
else:
    assert sm["ids_sha256"] == sm_drive["ids_sha256"] and \
        {k: v["sha256"] for k, v in sm["files"].items()} == {k: v["sha256"] for k, v in sm_drive["files"].items()}, \
        "ayrım ilk kayıttan farklı"
assert sm["seed"] == SPLIT_SEED and sm["train_source_sha256"] == env["train_ann_sha256"]
print(json.dumps(sm["distribution_check"], indent=1))
print({k: (v["images"], v["annotations"]) for k, v in sm["files"].items()})
# MÜHÜR: B dosyası bu notebook'ta hiçbir yükleyiciye verilmez (aşağıdaki kodda yalnız A, A_fit1k, bn2k kullanılır).

In [ ]:
# 7 · Öz-denetim + pilot + GPU kesinti/kurtarma testi (≈ 1–1,5 sa, ≈ 2 CU). Kayıtlar _pilot/ altında.
# (a) check_supernet.py (eşdeğerlik, çıkarma, sandwich, NSGA-II, ONNX) + check_resume.py (örnekleyici, artırma, depo)
#     + check_trainer_cpu.py (trainer CPU'da: kayıt yazılırken kesinti → devam, ağırlıklar birebir).
# (b) Panel: 50 sabit alt ağ (vekil gecikmeye göre 5 dilim) + 4 referans; başlangıç (epoch 0) fit1k değerleri.
# (c) Maliyet: 10k alt kümede 1 epoch (benchmark açık) → kararlı iter süresi → tam koşu tahmini.
# (d) Kesinti testi (deterministic cudnn, 10k, 2 epoch, kayıt her 50 iter, ilk 312 iterasyonun izi):
#     REF kesintisiz; TEST: 1. deneme 100. iterasyonun kaydı YAZILIRKEN öldürülür → 2. deneme 50'den devam,
#     ≥ 230. iterasyonda kill -9 → 3. deneme 206'dan devam (kayıtlar: 50, 100, 150, 156, 206, 256, 306, 312) → bitiş. İzler (alt ağ, lr, boyut, batch sha256) birebir
#     karşılaştırılır; kayıplar ve son ağırlıklar toleransla raporlanır.
import torch, re, time, signal
for script in ("check_supernet.py", "check_resume.py", "check_trainer_cpu.py"):
    r = subprocess.run(["python", f"{WS}/scripts/{script}"] + (["--weights", WEIGHTS] if script == "check_supernet.py" else []),
                       capture_output=True, text=True, cwd=WS)
    print(r.stdout[-1500:]); assert "GEÇTİ" in r.stdout.splitlines()[-1], r.stderr[-3000:]

BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"
EV = ss.Evaluator(COCO_LOCAL, batch_size=64, workers=WORKERS)
init_exp = sn.Exp(); init_exp.nas_pretrained = WEIGHTS
_sup = init_exp.get_model()
NAMES = init_exp.dw_names; N = len(NAMES)
REF_CFGS = {"nano_416": sn.nano_config(N, 416), "nano_320": sn.nano_config(N, 320),
            "max": sn.max_config(N), "min": sn.min_config(N)}
panel_path = f"{DRIVE_RESULTS}/panel.json"
if not os.path.exists(panel_path):
    ns.write_json(panel_path, {"panel": ss.make_panel(_sup, NAMES, n=50, groups=5, seed=2026),
                               "refs": {k: sn.config_id(v) for k, v in REF_CFGS.items()},
                               "method": "2000 tohumlu rastgele canonical alt ağ, ProxyLatency (DW MAC × çöz²) sırası, "
                                         "5 eşit dilim × 10 (seed 2026)", "created_utc": ns.now_utc()})
PANEL = ns.read_json(panel_path)
EVAL_SET = {**{k: sn.config_id(v) for k, v in REF_CFGS.items()}, **{p["config_id"]: p["config_id"] for p in PANEL["panel"]}}

def run_train(opts, tag, env_extra=None, kill_at=None):
    """train.py'yi alt süreçte koşar; kill_at verilirse log'daki ilerleme ≥ kill_at olunca SIGKILL."""
    cmd = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
           "-expn", tag] + opts
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=WS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, **(env_extra or {})})
    lines = []
    for line in proc.stdout:
        lines.append(line)
        m = re.search(r"epoch: (\d+)/\d+, iter: (\d+)/(\d+)", line)
        if m or re.search(r"devam:|doğrulanmış kayıt|\[test\]|Traceback|Error", line):
            print(f"[{tag} {time.time() - t0:5.0f}s] {line.strip()[-160:]}", flush=True)
        if kill_at is not None and m and (int(m.group(1)) - 1) * int(m.group(3)) + int(m.group(2)) >= kill_at:
            proc.send_signal(signal.SIGKILL); print(f"[{tag}] SIGKILL @ ≥{kill_at}", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines), time.time() - t0

RUN_PILOT = not os.path.exists(f"{DRIVE_RESULTS}/resume_test.json")
if RUN_PILOT:
    PILOT = f"{DRIVE_RESULTS}/_pilot"
    shutil.rmtree(PILOT, ignore_errors=True); shutil.rmtree(f"{OUT_ROOT}/_pilot", ignore_errors=True)
    # (b) başlangıç değerleri
    sup0 = _sup.cuda().eval()
    init_vals = {}
    for tag, cid in EVAL_SET.items():
        init_vals[tag] = ss.evaluate_config(sup0, NAMES, sn.parse_config_id(cid), EV, log=lambda s: None)["map5095"]
    print("epoch 0 referanslar:", {k: init_vals[k] for k in REF_CFGS})
    del sup0; torch.cuda.empty_cache()
    base10k = ["nas_pretrained", WEIGHTS, "output_dir", f"{OUT_ROOT}/_pilot", "data_num_workers", str(WORKERS),
               "train_ann", "instances_train2017_10k.json"]
    # (c) maliyet
    rc, log, wall = run_train(base10k + ["max_epoch", "1", "nas_phase1_epochs", "1", "no_aug_epochs", "-1",
                                         "nas_ckpt_every_iters", "0"], "_pilot/timing")
    assert rc == 0, log[-3000:]
    it = [float(x) for x in re.findall(r"iter_time: ([\d.]+)", log)]
    half = it[len(it) // 2:]; iter_s = sorted(half)[len(half) // 2]
    iters_full = 118_287 // BATCH
    pilot = {"iter_s_median_2nd_half": iter_s, "iters_per_epoch_full": iters_full, "epoch_s_est": round(iter_s * iters_full),
             "timing_wall_s": round(wall), "nas_r2_epoch_s_single_net": 886, "measured_utc": ns.now_utc(), "gpu": env["gpu"],
             "init_fit": init_vals}
    for E in (60, 90):
        h = iter_s * iters_full * E / 3600
        pilot[f"est_{E}ep"] = {"hours": round(h, 1), "cu": round(h * CU_PER_HOUR, 1)}
    ns.write_json(f"{DRIVE_RESULTS}/pilot.json", pilot)
    print(json.dumps({k: v for k, v in pilot.items() if k != "init_fit"}, indent=1))
    # (d) kesinti testi
    common = base10k + ["max_epoch", "2", "nas_phase1_epochs", "2", "no_aug_epochs", "0", "nas_ckpt_every_iters", "50",
                        "nas_deterministic", "True", "nas_trace_iters", "312"]
    def opts_for(name):
        return common + ["nas_ckpt_dir", f"{PILOT}/{name}/ckpt", "nas_trace_path", f"{PILOT}/{name}/trace.jsonl",
                         "nas_log_dir", f"{PILOT}/{name}"]
    for name in ("ref", "test"):
        os.makedirs(f"{PILOT}/{name}", exist_ok=True)
    rc, log, _ = run_train(opts_for("ref"), "_pilot/ref"); assert rc == 0, log[-3000:]
    rc1, log1, _ = run_train(opts_for("test"), "_pilot/test1", env_extra={"NAS_TEST_KILL_AT_SAVE": "100"})
    assert rc1 == 137, ("kayıt yazılırken kesinti beklenirdi", rc1, log1[-2000:])
    rc2, log2, _ = run_train(opts_for("test"), "_pilot/test2", kill_at=230)
    assert rc2 == -9 and "ilerleme 50)" in log2, ("50'den devam + SIGKILL beklenirdi", rc2, log2[-2000:])
    rc3, log3, _ = run_train(opts_for("test"), "_pilot/test3"); assert rc3 == 0, log3[-3000:]
    assert "ilerleme 206)" in log3, "son doğrulanmış kayıt (206) seçilmedi"
    cmp_ = nd.compare_traces(f"{PILOT}/ref/trace.jsonl", f"{PILOT}/test/trace.jsonl")
    from ckpt_store import CheckpointStore
    a = CheckpointStore(f"{PILOT}/ref/ckpt", "/tmp/ck_a").latest()[0]
    b = CheckpointStore(f"{PILOT}/test/ckpt", "/tmp/ck_b").latest()[0]
    assert a["progress"] == b["progress"] == 312
    wdiff = max((a["model_raw"][k].float() - b["model_raw"][k].float()).abs().max().item() for k in a["model_raw"])
    wscale = max(a["model_raw"][k].float().abs().max().item() for k in a["model_raw"])
    rt = {**cmp_, "final_weight_max_abs_diff": wdiff, "final_weight_max_abs": wscale,
          "attempts": {"test1_kill_during_save_rc": rc1, "test2_sigkill_rc": rc2, "test3_rc": rc3},
          "deterministic_cudnn": True, "tested_utc": ns.now_utc()}
    print(json.dumps(rt, indent=1, default=str))
    assert rt["n_exact_mismatches"] == 0 and not rt["missing_in_test"], "kesinti testi: birebir alanlarda fark var"
    ns.write_json(f"{DRIVE_RESULTS}/resume_test.json", rt)
else:
    print("pilot + kesinti testi kayıtlı:", ns.read_json(f"{DRIVE_RESULTS}/pilot.json")["est_60ep"],
          {k: ns.read_json(f"{DRIVE_RESULTS}/resume_test.json")[k] for k in ("n_exact_mismatches", "max_loss_rel_dev",
                                                                             "final_weight_max_abs_diff")})
del _sup

In [ ]:
# 8 · Dondurulmuş manifest: tarif, uzay, değerlendirme, erken kontrol, uzatma kuralı, mühür, test-dev durumu.
# Parmak izi alanları ilk yazımdan sonra değişirse hata verir (yeni tur = yeni ROUND).
from yolox.exp import get_exp
MAX_EPOCH, PHASE1, SNAP, PAUSE_EPOCH = 60, 60, 10, 4
EARLY_MAX_DROP = 3.0
TESTDEV_STATUS = "doğrulanmadı"   # ← Furkan/ajan: CodaLab gönderim kabulü denenince "çalışıyor"/"çalışmıyor" + tarih
for _m in ("supernet", "nano_macro"):
    sys.modules.pop(_m, None)
_e = get_exp(EXP_FILE, None)
_e.merge(["max_epoch", str(MAX_EPOCH), "nas_phase1_epochs", str(PHASE1), "nas_snapshot_interval", str(SNAP)])
RECIPE_KEYS = ("max_epoch", "nas_phase1_epochs", "no_aug_epochs", "warmup_epochs", "warmup_lr", "basic_lr_per_img",
               "min_lr_ratio", "scheduler", "weight_decay", "momentum", "ema", "input_size", "random_size",
               "mosaic_prob", "mosaic_scale", "enable_mixup", "mixup_prob", "hsv_prob", "flip_prob", "degrees",
               "translate", "shear", "depth", "width", "num_classes", "train_ann", "nmsthre",
               "nas_sandwich_random", "nas_seed", "nas_seed_data", "nas_ext_peak_ratio", "nas_ckpt_every_iters",
               "nas_snapshot_interval")
pilot = ns.read_json(f"{DRIVE_RESULTS}/pilot.json")
FINGERPRINT = ("round", "recipe", "exp_fingerprint", "space", "eval_protocol", "panel_sha256", "early_check",
               "extension_rule", "splits", "exp_file_sha256", "weights_sha256", "train_ann_sha256", "yolox_commit")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND, "notebook": "notebooks/03_gercek_nas_supernet_egitimi.ipynb",
    "decisions": "Karar 6 (Furkan, 2026-10-16, dondurulmuş tasarım): sandwich max+min+2 rastgele, inplace "
                 "distillation yok (BigNAS'tan açık sapma), 60 ep ilk aşama, val2017 A/B, B mühürlü.",
    "runs": ["supernet"],
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16, "devices": 1,
               "init": "pretrained yolox_nano.pth (3×3 → 7×7 merkez, kenar sıfır)", "train_images": 118_287},
    "exp_fingerprint": _e.fingerprint(),
    "space": {"kernels": sn.KERNELS, "depth_stages": sn.DEPTH_STAGES, "depths": sn.DEPTHS,
              "resolutions": sn.RESOLUTIONS, "dw_layers": NAMES},
    "eval_protocol": {"split": "val2017 A_fit1k (1000)", "bn_recalibration": "train2017 bn2k (2000), kümülatif",
                      "precision": "fp16", "conf": 0.001, "nms": 0.65, "checkpoint": "EMA (kayıttaki 'model')",
                      "ap_scale": "0–100", "snapshots": [0, 20, 30, 40, 50, 60], "refs": PANEL["refs"]},
    "panel_sha256": ns.sha256(panel_path),
    "early_check": {"epoch": PAUSE_EPOCH, "metric": "nano_416 fit1k mAP", "init_value": pilot["init_fit"]["nano_416"],
                    "max_drop": EARLY_MAX_DROP, "on_fail": "dur, tarif konuşulur"},
    "extension_rule": {
        "type": "önceden seçilmiş mühendislik eşikleri (literatür yeterlilik eşiği değil)",
        "evaluate_at": 60,
        "extend_if": "panel ortalaması e50→e60 ≥ +0,3 AP (hâlâ iyileşiyor) VE hiçbir grup ortalaması < −0,3",
        "stop_if": "panel ortalaması e50→e60 < +0,3 AP VE τ(e40, e60) ≥ 0,8",
        "investigate_if": "panel ortalaması düşüyor (< −0,3) ya da τ(e40, e60) < 0,8 iken iyileşme yok → örnekleme/"
                          "ağırlık paylaşımı sorgulanır; otomatik uzatma yok",
        "extension": {"epochs": 30, "max_extensions": 1, "optimizer": "korunur",
                      "peak_lr": "0,5 × ilk aşama tepe lr (0,001/64 × 64)", "warmup": 0,
                      "schedule": "cosine → min_lr_ratio × tepe; son no_aug_epochs+1 epoch mosaic'siz, lr min",
                      "mosaic": "yeniden açık (epoch 60–87), 88–89 kapalı"},
        "reported": "ortalama değişim, grup ortalamaları, > 0,3 düşen aday sayısı, en büyük düşüş, τ genel + grup içi",
        "approval": "uzatma da uzun koşu: Furkan onayı"},
    "splits": {"seed": sm["seed"], "ids_sha256": sm["ids_sha256"],
               "sealed": "val2017 B: NAS + budama + nicemleme kararları bitene kadar açılmaz; önçalışmada (nas_r1b/r2) "
                         "tam val2017 dolaylı kullanıldı (tezde yazılır)"},
    "final_eval": {"primary": "COCO test-dev (gönderim kabulü doğrulanırsa; yöntem + checkpoint dondurulduktan sonra, "
                              "sonuca göre ayar yok)", "fallback": "val2017 B (sınırlamayla)",
                   "testdev_status": TESTDEV_STATUS,
                   "known_issue": "cocoapi'de 2026-01-21 tarihli 'yeni gönderim kabul edilmiyor' açık sorunu"},
    "resume_test": ns.read_json(f"{DRIVE_RESULTS}/resume_test.json"),
    "exp_file_sha256": ns.sha256(EXP_FILE), "weights_sha256": env["weights_sha256"],
    "weights_url": env["weights_url"], "train_ann_sha256": env["train_ann_sha256"],
    "yolox_commit": env["yolox_commit"], "repo_commit_at_creation": env["repo_commit"],
    "cu_per_hour": CU_PER_HOUR, "pilot": pilot, "env_at_creation": env,
}, keys=FINGERPRINT)
del _e
assert manifest["resume_test"]["n_exact_mismatches"] == 0
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "early_check", "final_eval")}, indent=1, default=str))
print("tahmin:", manifest["pilot"]["est_60ep"], "· test-dev:", manifest["final_eval"]["testdev_status"])

In [ ]:
# 9 · Süper ağ eğitimi (UZUN KOŞU — Furkan'ın başlatma onayıyla). Her başlatmada en yeni doğrulanmış kayıttan devam.
# Epoch PAUSE_EPOCH sonunda kendini durdurur → hücre 10 erken kontrol → bu hücre yeniden çalıştırılınca devam.
RUN = "supernet"
MIRROR = f"{DRIVE_RESULTS}/runs/{RUN}"
CKPT_DIR = f"{MIRROR}/ckpt"
from ckpt_store import CheckpointStore
def store():
    return CheckpointStore(CKPT_DIR, f"{OUT_ROOT}/{RUN}/ck_local", log=print)

def latest_progress():
    recs = store().records() if os.path.isdir(CKPT_DIR) else []
    return recs[-1]["progress"] if recs else None

def train_supernet():
    st = ns.get_status(DRIVE_RESULTS, RUN)
    if st["status"] in ("trained", "evaluated"):
        return st
    os.makedirs(MIRROR, exist_ok=True)
    ipe = 118_287 // BATCH
    p = latest_progress()
    if p is None and st["attempts"]:
        print("uyarı: önceki deneme var ama doğrulanmış kayıt yok → baştan")
    if p is not None and p >= PAUSE_EPOCH * ipe and not st.get("early_check_passed"):
        raise RuntimeError("erken kontrol (hücre 10) henüz geçmedi")
    if p is None or p < MAX_EPOCH * ipe:
        pause = PAUSE_EPOCH if (p or 0) < PAUSE_EPOCH * ipe else 0
        attempt = {"start_utc": ns.now_utc(), "resume_from_progress": p, "pause_epoch": pause, "gpu": env["gpu"],
                   "repo_commit": env["repo_commit"]}
        st = ns.set_status(DRIVE_RESULTS, RUN, st, status="training", attempts=st["attempts"] + [attempt])
        rc, log, wall = run_train(["nas_pretrained", WEIGHTS, "nas_ckpt_dir", CKPT_DIR, "nas_log_dir", MIRROR,
                                   "output_dir", OUT_ROOT, "data_num_workers", str(WORKERS),
                                   "max_epoch", str(MAX_EPOCH), "nas_phase1_epochs", str(PHASE1),
                                   "nas_snapshot_interval", str(SNAP), "nas_pause_epoch", str(pause)], RUN)
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1), rc=rc)
        st = ns.set_status(DRIVE_RESULTS, RUN, st)
        assert rc == 0, f"eğitim çıkış kodu {rc}: hücreyi yeniden çalıştır (kayıttan devam eder)\n{log[-2000:]}"
        p = latest_progress()
        if p < MAX_EPOCH * ipe:
            print(f"duraklatıldı: ilerleme {p} (epoch {p // ipe}) → hücre 10 (erken kontrol)")
            return st
    log = open(f"{MIRROR}/train_log.txt").read()
    ns.check_train_log(log, MAX_EPOCH, 1)
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, RUN, st, status="trained", train_s=train_s, trained_utc=ns.now_utc())

st = train_supernet()
print(RUN, st["status"], "deneme", len(st["attempts"]), "eğitim", st.get("train_s"))

In [ ]:
# 10 · Anlık görüntü değerlendirmesi (fit1k, BN kalibreli) + erken kontrol + uzatma ölçütü raporu.
# epoch 4: yalnız referanslar (erken kontrol); 20/30/40/50/60: referanslar + 50'lik panel. Her sonuç status'a yazılır.
import csv
ipe = 118_287 // BATCH
def snapshot_eval(epoch, which):
    st = ns.get_status(DRIVE_RESULTS, RUN)
    key = f"e{epoch}"
    done = st.get("snapshots", {}).get(key, {})
    state, rec = store().find(epoch * ipe)
    assert state is not None, f"epoch {epoch} kaydı yok"
    sup = None
    for tag, cid in which.items():
        if tag in done:
            continue
        if sup is None:
            e_ = sn.Exp(); sup = e_.get_model(); sup.load_state_dict(state["model"], strict=True); sup = sup.cuda().eval()
        done[tag] = ss.evaluate_config(sup, NAMES, sn.parse_config_id(cid), EV, log=lambda s: None)["map5095"]
        st = ns.set_status(DRIVE_RESULTS, RUN, st, snapshots={**st.get("snapshots", {}), key: done})
    del sup, state; torch.cuda.empty_cache()
    return ns.get_status(DRIVE_RESULTS, RUN)

refs = {k: v for k, v in EVAL_SET.items() if k in REF_CFGS}
recs = {r["progress"] for r in store().records()}
for e in [PAUSE_EPOCH] + list(range(20, MAX_EPOCH + 1, SNAP)):
    if e * ipe in recs:
        st = snapshot_eval(e, refs if e == PAUSE_EPOCH else EVAL_SET)
        print(f"epoch {e}:", {k: st["snapshots"][f"e{e}"].get(k) for k in REF_CFGS})

st = ns.get_status(DRIVE_RESULTS, RUN)
snaps = {0: manifest["pilot"]["init_fit"], **{int(k[1:]): v for k, v in st.get("snapshots", {}).items()}}
rows = [{"epoch": e, "subnet": t, "map5095": v} for e, d in sorted(snaps.items()) for t, v in d.items()]
with open(f"{DRIVE_RESULTS}/snapshots.csv.tmp", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["epoch", "subnet", "map5095"]); w.writeheader(); w.writerows(rows)
os.replace(f"{DRIVE_RESULTS}/snapshots.csv.tmp", f"{DRIVE_RESULTS}/snapshots.csv")

if PAUSE_EPOCH in snaps and not st.get("early_check_passed"):
    v, ref = snaps[PAUSE_EPOCH]["nano_416"], manifest["early_check"]["init_value"]
    print(f"ERKEN KONTROL: nano_416 fit1k {v} (başlangıç {ref}, izin verilen düşüş {EARLY_MAX_DROP})")
    assert v >= ref - EARLY_MAX_DROP, "süper ağ önceden eğitilmiş bilgiyi bozuyor: dur, tarif konuşulmalı"
    ns.set_status(DRIVE_RESULTS, RUN, st, early_check_passed=True, early_check={"value": v, "ref": ref})
    print("erken kontrol geçti → hücre 9'u yeniden çalıştır")

panel = PANEL["panel"]
pairs = [(a, b) for a, b in ((0, 20), (20, 30), (30, 40), (40, 50), (50, 60), (40, 60)) if a in snaps and b in snaps]
pstats = {f"e{a}->e{b}": ss.panel_stats(snaps[a], snaps[b], panel) for a, b in pairs}
ns.write_json(f"{DRIVE_RESULTS}/panel_stats.json", pstats)
for k, v in pstats.items():
    print(k, {x: v[x] for x in ("mean_change", "n_drop_gt", "max_drop", "tau_all")},
          {g: (s["mean_change"], s["tau"]) for g, s in v["groups"].items()})
if 50 in snaps and 60 in snaps and 40 in snaps:
    m = pstats["e50->e60"]; t = pstats["e40->e60"]["tau_all"]
    groups_ok = all(s["mean_change"] >= -0.3 for s in m["groups"].values())
    if m["mean_change"] >= 0.3 and groups_ok:
        verdict = "UZAT (önceden yazılı ölçüt): +30 epoch — Furkan onayıyla MAX_EPOCH=90"
    elif m["mean_change"] < 0.3 and t >= 0.8 and m["mean_change"] >= -0.3:
        verdict = "DUR: süper ağ hazır → 03b arama"
    else:
        verdict = "İNCELE: düşüş ya da kararsız sıralama; otomatik uzatma yok"
    print("UZATMA ÖLÇÜTÜ:", verdict, "| e50→e60", m["mean_change"], "| τ(e40,e60)", t)
    ns.set_status(DRIVE_RESULTS, RUN, ns.get_status(DRIVE_RESULTS, RUN), extension_verdict=verdict)

In [ ]:
# 11 · Özet: anlık görüntü eğrisi (referanslar + panel grup ortalamaları) + süre/CU → summary.json, png
import pandas as pd, matplotlib.pyplot as plt
st = ns.get_status(DRIVE_RESULTS, RUN)
df = pd.read_csv(f"{DRIVE_RESULTS}/snapshots.csv")
grp = {p["config_id"]: f"panel_g{p['group']}" for p in PANEL["panel"]}
df["series"] = df["subnet"].map(lambda s: grp.get(s, s))
piv = df.groupby(["epoch", "series"])["map5095"].mean().unstack()
print(piv.round(2).to_string())
cu = (st.get("train_s") or sum(a.get("wall_s") or 0 for a in st["attempts"])) / 3600 * CU_PER_HOUR
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {
    "round": ROUND, "written_utc": ns.now_utc(), "status": st["status"], "train_s": st.get("train_s"),
    "cu_train": round(cu, 2), "attempts": st["attempts"], "early_check": st.get("early_check"),
    "extension_verdict": st.get("extension_verdict"), "panel_stats": ns.read_json(f"{DRIVE_RESULTS}/panel_stats.json")})
fig, ax = plt.subplots(figsize=(8, 4.8))
for col in piv.columns:
    ax.plot(piv.index, piv[col], marker="o", label=col, ls="-" if col in REF_CFGS else "--")
ax.set_xlabel("epoch (süper ağ)"); ax.set_ylabel("mAP@[.5:.95] (0–100), val2017 A_fit1k, BN kalibreli")
ax.set_title("nas_super1: referans alt ağlar ve panel dilimleri"); ax.legend(fontsize=7, ncol=2); fig.tight_layout()
fig.savefig(f"{DRIVE_RESULTS}/{ROUND}_snapshots.png", dpi=150); plt.show()
print(f"eğitim ≈ {cu:.1f} CU")